### RAG Pipelines _ Data Ingestion to Vector DB Pipeline

In [3]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\deept\AppData\Local\Temp\ipykernel_16528\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
c:\Users\deept\Documents\RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
### Read all the PDF's inside the directory
def process_all_pdfs_in_Directory(pdf_directory):
    """Process all the PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    #Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("*/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages from {pdf_file.name}")

        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents    

#Process all PDFs in the specified directory
all_pdf_documents = process_all_pdfs_in_Directory("../data")

Found 10 PDF files to process

Processing: Deeptesh's Cognifyz certificate.pdf
 Loaded 1 pages from Deeptesh's Cognifyz certificate.pdf

Processing: Deeptesh's Oracle Certificate.pdf
 Loaded 1 pages from Deeptesh's Oracle Certificate.pdf

Processing: Deeptesh's Prodigy Infotech Certificate.pdf
 Loaded 1 pages from Deeptesh's Prodigy Infotech Certificate.pdf

Processing: My ML course 1 Certificate.pdf
 Loaded 1 pages from My ML course 1 Certificate.pdf

Processing: My ML course 2 Certificate.pdf
 Loaded 1 pages from My ML course 2 Certificate.pdf

Processing: My ML course 3 Certificate.pdf
 Loaded 1 pages from My ML course 3 Certificate.pdf

Processing: My NISM Certificate.pdf
 Loaded 1 pages from My NISM Certificate.pdf

Processing: NCAT Participation Certificate.pdf
 Loaded 1 pages from NCAT Participation Certificate.pdf

Processing: Prodigy Infotech Letter of Recommendation.pdf
 Loaded 1 pages from Prodigy Infotech Letter of Recommendation.pdf

Processing: RAG_Explanation.pdf
 Loaded

In [5]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m145', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/145.0.0.0 Safari/537.36', 'creationdate': '2026-07-15T05:39:32+00:00', 'title': 'Deeptesh Sagar', 'moddate': '2026-07-15T05:39:32+00:00', 'source': "..\\data\\pdf\\Deeptesh's Cognifyz certificate.pdf", 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': "Deeptesh's Cognifyz certificate.pdf", 'file_type': 'pdf'}, page_content='Date\xa0-15/07/2026\xa0\nThis\xa0is\xa0to\xa0certify\xa0that\xa0Deeptesh\xa0 Sagar\xa0 (Intern\xa0 ID:\xa0 CTI/A1/C373903)\xa0 currently\xa0pursuing\xa0\nB.Tech\xa0from\xa0The\xa0Sanjeev\xa0Agrawal\xa0Global\xa0Educational\xa0University,\xa0Bhopal,\xa0was\xa0working\xa0\nas\xa0a\xa0 Machine\xa0 Learning\xa0Intern\xa0with\xa0Cognifyz\xa0IT\xa0Solutions\xa0Pvt.\xa0Ltd.\xa0from\xa0 15/06/2026\xa0 to\xa0\n15/07/2026.\nDuring\xa0 this\xa0 period,\xa0 he\xa0 has\xa0 served\xa0 as\xa0 a\xa0 Machine\xa0 Learning\xa0 

In [6]:
###Text Splitting get into chunks

def split_documents_into_chunks(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into small chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    #Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:\n" f"Content: {split_docs[0].page_content[:500]}...\n" f"Metadata: {split_docs[0].metadata}")
    return split_docs


In [7]:
chunks=split_documents_into_chunks(all_pdf_documents)
chunks

Split 10 documents into 13 chunks

Example chunk:
Content: Date -15/07/2026 
This is to certify that Deeptesh  Sagar  (Intern  ID:  CTI/A1/C373903)  currently pursuing 
B.Tech from The Sanjeev Agrawal Global Educational University, Bhopal, was working 
as a  Machine  Learning Intern with Cognifyz IT Solutions Pvt. Ltd. from  15/06/2026  to 
15/07/2026.
During  this  period,  he  has  served  as  a  Machine  Learning  Intern  and  has  displayed 
remarkable  dedication,  sincerity,  and  a  strong  desire  to  learn.  He  has  exhibited 
exceptional  coo...
Metadata: {'producer': 'Skia/PDF m145', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/145.0.0.0 Safari/537.36', 'creationdate': '2026-07-15T05:39:32+00:00', 'title': 'Deeptesh Sagar', 'moddate': '2026-07-15T05:39:32+00:00', 'source': "..\\data\\pdf\\Deeptesh's Cognifyz certificate.pdf", 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': "Deeptesh's Cognifyz certificate.p

[Document(metadata={'producer': 'Skia/PDF m145', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/145.0.0.0 Safari/537.36', 'creationdate': '2026-07-15T05:39:32+00:00', 'title': 'Deeptesh Sagar', 'moddate': '2026-07-15T05:39:32+00:00', 'source': "..\\data\\pdf\\Deeptesh's Cognifyz certificate.pdf", 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': "Deeptesh's Cognifyz certificate.pdf", 'file_type': 'pdf'}, page_content='Date\xa0-15/07/2026\xa0\nThis\xa0is\xa0to\xa0certify\xa0that\xa0Deeptesh\xa0 Sagar\xa0 (Intern\xa0 ID:\xa0 CTI/A1/C373903)\xa0 currently\xa0pursuing\xa0\nB.Tech\xa0from\xa0The\xa0Sanjeev\xa0Agrawal\xa0Global\xa0Educational\xa0University,\xa0Bhopal,\xa0was\xa0working\xa0\nas\xa0a\xa0 Machine\xa0 Learning\xa0Intern\xa0with\xa0Cognifyz\xa0IT\xa0Solutions\xa0Pvt.\xa0Ltd.\xa0from\xa0 15/06/2026\xa0 to\xa0\n15/07/2026.\nDuring\xa0 this\xa0 period,\xa0 he\xa0 has\xa0 served\xa0 as\xa0 a\xa0 Machine\xa0 Learning\xa0 

### Embedding and VectorStoreDB

In [8]:
import os
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [9]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager with a specified model.

        Args:
            model_name: HuggingFace model name for generating embeddings.
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """ Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:

        """
        Generate embeddings for a list of texts.

        Args:
            texts: List of text strings to embed 

        Returns:
            Numpy array of embedding with shape (len(texts), embedding_dim) """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar = True)
        print(f"generated embeddings with shape: {embeddings.shape}")
        return embeddings


    def get_embedding_dimension(self) -> int:
        """Get the dimension of the embeddings generated by the model."""
        if not self.model:
            raise ValueError("Model not loaded")
        return self.model.get_sentence_embedding_dimension()

## Initialize the Embedding Manager

embedding_manager = EmbeddingManager()
embedding_dimension = embedding_manager.get_embedding_dimension()
embedding_manager
embedding_dimension

    

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6775.30it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\deept\AppData\Local\Temp\ipykernel_16528\122420175.py:19: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
C:\Users\deept\AppData\Local\Temp\ipykernel_16528\122420175.py:47: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  return self.model.get_sentence_embedding_dimension()


384

### VectorStore

In [10]:
class VectorStore:
    """Manage Document embedding in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """ Initialize the ChromeDB client and collection"""
        try:
            # Create persistent ChromaDb Client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"Description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f" Error initializing vector store: {e}")
            raise
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of Langchain cosuments
            embeddings: Corressponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")


        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)


            #prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            #Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids, 
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 23


In [11]:
chunks

[Document(metadata={'producer': 'Skia/PDF m145', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/145.0.0.0 Safari/537.36', 'creationdate': '2026-07-15T05:39:32+00:00', 'title': 'Deeptesh Sagar', 'moddate': '2026-07-15T05:39:32+00:00', 'source': "..\\data\\pdf\\Deeptesh's Cognifyz certificate.pdf", 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': "Deeptesh's Cognifyz certificate.pdf", 'file_type': 'pdf'}, page_content='Date\xa0-15/07/2026\xa0\nThis\xa0is\xa0to\xa0certify\xa0that\xa0Deeptesh\xa0 Sagar\xa0 (Intern\xa0 ID:\xa0 CTI/A1/C373903)\xa0 currently\xa0pursuing\xa0\nB.Tech\xa0from\xa0The\xa0Sanjeev\xa0Agrawal\xa0Global\xa0Educational\xa0University,\xa0Bhopal,\xa0was\xa0working\xa0\nas\xa0a\xa0 Machine\xa0 Learning\xa0Intern\xa0with\xa0Cognifyz\xa0IT\xa0Solutions\xa0Pvt.\xa0Ltd.\xa0from\xa0 15/06/2026\xa0 to\xa0\n15/07/2026.\nDuring\xa0 this\xa0 period,\xa0 he\xa0 has\xa0 served\xa0 as\xa0 a\xa0 Machine\xa0 Learning\xa0 

In [12]:
### Convert text into embeddings
texts=[doc.page_content for doc in chunks]

## Generate embeddings for the chunks

embeddings=embedding_manager.generate_embeddings(texts)

## Store in the vector database
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 13 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.40it/s]

generated embeddings with shape: (13, 384)
Adding 13 documents to vector store...
Successfully added 13 documents to vector store
Total documents in collection: 36


### Retriever Pipeline From Vectorstore

In [13]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
            """ 
            Retrieve relevant documents for a query
                
            Args:
                query: The search query
                top_k: Number of top results to return
                score_threshold; Minimum similarity score thresholdfor filtering results
                    
            Returns:
                List of dictionaries containing retrieved documents and metadata
            """
            print(f"Retrieving documents for query:'{query}'")
            print(f"Top K: {top_k}, Score threshold: {score_threshold}")
    
                # Generate query embedding
            query_embedding = self.embedding_manager.generate_embeddings([query])[0]
    
            # Search in vectore store
            try:
                results = self.vector_store.collection.query(
                    query_embeddings=[query_embedding.tolist()], 
                    n_results=top_k
                )
    
                 # Process results
                retrieved_docs =  []

                if results['documents'] and results['documents'][0]:
                    documents = results['documents'][0]
                    metadatas = results['metadatas'][0]
                    distances = results['distances'][0]
                    ids = results['ids'][0]
    
    
                    for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                        #Convert distances to similarity scores (ChromaDb uses cosine disctance)
                        similarity_score = 1 - distance

                        if similarity_score >= score_threshold:
                            retrieved_docs.append({
                                'id': doc_id, 
                                'content': document,
                                'metadata': metadata,
                                    'similarity_score': similarity_score,
                                    'distance': distance,
                                    'rank': i + 1
                                    })
                                
                        print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
                    else:
                        print("No documents retrieved from vector store")
    
                    return retrieved_docs
            except Exception as e:
                    print(f"Error during retrievel: {e}")
                    return []

rag_retriever = RAGRetriever(vectorstore, embedding_manager)
            

In [14]:
rag_retriever

In [15]:
rag_retriever.retrieve("What is RAG?")

Retrieving documents for query:'What is RAG?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 90.35it/s]

generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
Retrieved 2 documents (after filtering)
Retrieved 3 documents (after filtering)
Retrieved 4 documents (after filtering)
Retrieved 5 documents (after filtering)
No documents retrieved from vector store


[{'id': 'doc_37e25d44_0',
  'content': 'RAG (Retrieval-Augmented Generation)\nRAG stands for Retrieval-Augmented Generation. It is a technique used with Large Language\nModels (LLMs) to make their responses more accurate, relevant, and based on specific\nexternal information.\nNormally, an LLM answers using the knowledge it learned during training. With RAG, the model can\nsearch an external knowledge source, such as PDFs, documents, databases, or websites, and\nuse the retrieved information to answer the user\'s question.\nHow RAG Works\n1\nUser Query – The user asks a question.\n2\nRetrieval – The system searches a knowledge base for relevant information.\n3\nAugmentation – The retrieved information is added to the user\'s query as context.\n4\nGeneration – The LLM uses this context to generate the final answer.\nExample\nSuppose a college has a PDF containing its rules and regulations. If a student asks, "What is the\nattendance requirement?", a RAG system searches the PDF, retrieve

### Integration VectorDB Context pipeline With LLM Output

In [16]:
### Simple RAG Pipeline with gemini LLM
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()

# 1. Initialize the Gemini LLM (set your gemini api key in the environment)
gemini_api_key = os.getenv("GEMINI_API_KEY")

llm=ChatGoogleGenerativeAI(model= "gemini-3.5-flash", api_key=gemini_api_key, temperature=0.2, max_output_tokens=512)

## 2. Simple RAG function: retrieve context and generate response
def rag_query(query, retriever, llm, top_k=5, score_threshold=0.0):
    """
    Perform a RAG query: retrieve relevant documents and generate a response using LLM

    Args:
        query: The search query
        retriever: RAGRetriever instance for document retrieval
        llm: Language model instance for response generation
        top_k: Number of top documents to retrieve
        score_threshold: Minimum similarity score threshold for filtering results

    Returns:
        Generated response from the LLM based on retrieved context
    """
    # Step 1: Retrieve relevant documents
    results = retriever.retrieve(query, top_k=top_k, score_threshold=score_threshold)

    if not results:
        return "No relevant documents found."

    # Step 2: Prepare context for LLM
    context = "\n\n".join([f"Document {doc['rank']}:\n{doc['content']}" for doc in results])

    # Step 3: Generate response using LLM
    prompt = f"""Use the following context to answer the question concisely.Context:\n{context}\n\nQuestion: {query}\nAnswer:"""
    
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    return response.content

In [17]:
answer=rag_query("What is RAG?", rag_retriever, llm, top_k=5, score_threshold=0.0)
print(answer)

Retrieving documents for query:'What is RAG?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 128.20it/s]
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
Retrieved 2 documents (after filtering)
Retrieved 3 documents (after filtering)
Retrieved 4 documents (after filtering)
Retrieved 5 documents (after filtering)
No documents retrieved from vector store
[{'type': 'text', 'text': "**RAG (Retrieval-Augmented Generation)** is a technique used with Large Language Models (LLMs) to make their responses more accurate and relevant. It works by searching external knowledge sources (such as PDFs, databases, or websites) for relevant information and using that retrieved content to help answer the user's question.", 'extras': {'signature': 'EoULCoILARFNMg/A82SEdne5YhWrteHTfNBEY7uQfZ7T8aafdiSABBjtN+MUFW0EVXh1LaXTLtebSwLyHEIvYVFt4KVYOVY0roWxLTR30hzrWBR9vR6UqI4dsVIveaOaQyD/jAzVdy1eRutdKVX+fQ4VESu3/q59pe0hSuch7msyWtUsk9mld0LanWsnEKqu70xGalzyaMbR/U313cF9EaTwcDIiSwcR/aLFpH5k+/SuvI5yJE1koW5Tr5rEEjQiDn+OitozA1z3E5e36TtxXZ1f/HQIQ9zV3ui6ybBm8wdA8pJZWrmru+rYxpG2P8U6k7HKP5vdC7i8Lj

### Enhanced RAG Pipeline Features

In [22]:
# Enhanced RAG Pipeline with gemini LLM and context-aware prompt engineering
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    Perform an advanced RAG query with context-aware prompt engineering.

    Args:
        query: The search query
        retriever: RAGRetriever instance for document retrieval
        llm: Language model instance for response generation
        top_k: Number of top documents to retrieve
        min_score: Minimum similarity score threshold for filtering results
        return_context: If True, return the retrieved context along with the answer

    Returns:
        Generated response from the LLM based on retrieved context, optionally with context
    """
    # Step 1: Retrieve relevant documents
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)

    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'content': ''}

    # Step 2: Prepare context for LLM
    context = "\n\n".join([f"Document {doc['rank']}:\n{doc['content']}" for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])


    # Step 3: Generate response using LLM with enhanced prompt
    prompt = f"""You are an expert assistant. Use the following context to answer the question concisely and accurately.\nContext:\n{context}\n\nQuestion: {query}\nAnswer:"""
    response = llm.invoke(prompt)

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence,
        'content': context
    }
    if return_context:
        output['context'] = context
    return output

#Example usage:
result = rag_advanced("which course did deeptesh do?", rag_retriever, llm, top_k=3, min_score=0.0, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['content'][:500], "...")

Retrieving documents for query:'which course did deeptesh do?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 59.21it/s]

generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
Retrieved 2 documents (after filtering)
Retrieved 2 documents (after filtering)
No documents retrieved from vector store


Answer: [{'type': 'text', 'text': 'Based on the provided context, there is no mention of any specific course that Deeptesh did. The text only mentions his internship and his work in the field of Machine Learning.', 'extras': {'signature': 'EqcKCqQKARFNMg+EKcp01HBZkPMe3Ruf/ywU4pHkvpTxE/tYjdneg7bMQFdi5nojlLQtQ2yUNB4w9XtJmwnYmwegtKbTCK+GghV8BlIY5htUP6B4PiVFuQTm9DLA3zstZ0fEaKeo9SyF4GnHZkDWLu8Rb8AtB+LY1QtukBKqf1XRjFMnbUnmqaiDS3AxjRnm1qmvEG30Zl0JtA6Teup9MrN/xNCUTi6To7QkkZG9XoHa7STQnIRAf38WsUqF865JGaw7LFrYudD+FXgELhVnVwJPDq3NWti32nI4SGjPmM6FQdSaPSY+E4/6+YVUQ0JGUn7gch9en0smEEy0Gufyl1sRs0s6uGy0yy5p2+uD+6J43IrthOugv8VGX0iQlXZe8FLG99NP3kBopaUxOPgcvKPuJsYUeacKWXMz3KHPpPFn/FBSoyyVVNk3Z+oI+EMQgtjgkxOVBqSfsowvPIUXCqISXSe9VElgQkMHXlk7+skO5v5qYoRWKVV8aVUdxv5OjcOfFsl59FRhCtEEAq71hYCShPPtBPOlHVbeHNkpCRMv72SfuJZLv2SVsQjSgI41AVRbvqI3br0ZBKsZUN1nK0tfSiszX7qCzTQMPaZ8BwGfOLbpYw60DnliEUDRi/3ufQZFcnxHMQAhBxP/Tm1u6QR/HQxTaBqoBdNDxH68cYRAAxK9lrV8T5FfeTOAasRQL6eAgztxi/DWarsJMBf6FdYNehMT3Zb/vRgzCtUU3HZQxjzyxaUHlXCC

In [23]:
#Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        """
        Initialize the Advanced RAG Pipeline

        Args:
            retriever: RAGRetriever instance for document retrieval
            llm: Language model instance for response generation
        """
        self.retriever = retriever
        self.llm = llm
        self.history = []  # To maintain conversation history
    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        #Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'content': ''}
        else:
            context = "\n\n".join([f"Document {doc['rank']}:\n{doc['content']}" for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:300] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""You are an expert assistant. Use the following context to answer the question concisely and accurately.\nContext:\n{context}\n\nQuestion: {question}\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.1)  # Simulate streaming delay
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        #Add citations to answer
        citation = [f'[{i+1}] {src["source"]} (page {src["page"]})' for i, src in enumerate(sources)]
        answer_with_citations = f"{answer}\n\nCitations:\n" + "\n".join(citation) if citation else answer

        #Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_response = self.llm.invoke([summary_prompt])
            summary = summary_response.content

        #Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary,
        })

        return{
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }
#Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("which course did deeptesh do?", top_k=3, min_score=0.0, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query:'which course did deeptesh do?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 115.47it/s]

generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
Retrieved 2 documents (after filtering)
Retrieved 2 documents (after filtering)
No documents retrieved from vector store
Streaming answer:
You are an expert assistant. Use the following context to answer the question concisely and accurately.
Context:
Document 1:
acquire new knowledge and skills ha

ve been evident throughout His internship.
We extend our best wishes to Deeptesh Sagar  for  a  successful  future,  and  we  have 
no doubt that he will continue to excel in the field of Machine Learning.

Document 2:
acquire new knowledge and skills have been evident throughout His internship.
We extend our best wishes to Deeptesh Sagar  for  a  successful  future,  and  we  have 
no doubt that he will continue to excel in the field of Machine Learning.

Question: which course did deeptesh do?
Answer:

Final Answer: [{'type': 'text', 'text': 'Based on the provided context, there is no mention of a specific course. However, it indicates that he completed an **internship** in the field of **Machine Learning**.', 'extras': {'signature': 'EtULCtILARFNMg8nI8b5QSFVxVW8jMQOjSw83tlBOH7tcqkXIGI4fTvUIIdCE7cpUalZXwjCJww6R3ypeDdYVS013jbt/Vo4JJBF1IE/dTcqvKKZD6u5MfCrjl+BDFLVAtqXOh8KqF77te1h0t0n4xQ+zkV4d1FN3vuXecommgaZDPdPeDj5+QAqKB73AHWDH4NXny1h/UzTfarvf+eRTiwvFS0zkamrB7ym2KuNmsC/4O1xVYsr2E+ar5ir4